In [1]:
import pandas as pd
import math

In [2]:
df = pd.read_csv("play_tennis.csv")
df = df.drop("day", axis=1)
print(df)

     outlook  temp humidity    wind play
0      Sunny   Hot     High    Weak   No
1      Sunny   Hot     High  Strong   No
2   Overcast   Hot     High    Weak  Yes
3       Rain  Mild     High    Weak  Yes
4       Rain  Cool   Normal    Weak  Yes
5       Rain  Cool   Normal  Strong   No
6   Overcast  Cool   Normal  Strong  Yes
7      Sunny  Mild     High    Weak   No
8      Sunny  Cool   Normal    Weak  Yes
9       Rain  Mild   Normal    Weak  Yes
10     Sunny  Mild   Normal  Strong  Yes
11  Overcast  Mild     High  Strong  Yes
12  Overcast   Hot   Normal    Weak  Yes
13      Rain  Mild     High  Strong   No


In [3]:
def entropy(data):
    target = data.iloc[:, -1]
    counts = target.value_counts()
    ent = 0
    for c in counts:
        p = c / len(target)
        ent -= p * math.log2(p)
    return ent

In [4]:
def information_gain(data, attribute):
    total = entropy(data)
    weighted = 0
    for value in data[attribute].unique():
        subset = data[data[attribute] == value]
        weighted += (len(subset) / len(data)) * entropy(subset)
    return total - weighted


In [11]:
def id3(data, attributes):
    target = data.iloc[:, -1]

    if target.nunique() == 1:        # pure group - stop
        return target.iloc[0]
    if len(attributes) == 0:         # no questions left - majority vote
        return target.mode()[0]

    gains = {}                       # <-- 4 spaces indentation + curly braces {}
    for a in attributes:
        gains[a] = information_gain(data, a)
    best = max(gains, key=gains.get)
    tree = {best: {}}
    for value in data[best].unique():
        subset = data[data[best] == value]
        remaining = [a for a in attributes if a != best]
        tree[best][value] = id3(subset, remaining)
    return tree

In [12]:
def predict(tree, sample):
    if not isinstance(tree, dict):          # reached an answer
        return tree
    attribute = list(tree.keys())[0]
    subtree = tree[attribute][sample[attribute]]
    return predict(subtree, sample)

In [13]:
attributes = list(df.columns[:-1])
tree = id3(df, attributes)
print(tree)
print(predict(tree, {"outlook": "Sunny", "temp": "Cool", "humidity": "High", "wind": "Strong"}))

{'outlook': {'Sunny': {'humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'wind': {'Weak': 'Yes', 'Strong': 'No'}}}}
No
